# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-07-31T11:57:07.653Z

## 1. Install dependencies

In [1]:
%pip install autora-theorist-darts==1.1.0 autora-core==5.0.3 autora-experiment-runner-firebase-prolific==1.0.1

  Using cached autora_experiment_runner_firebase_prolific-1.0.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached autora_experiment_runner_recruitment_manager_prolific-1.0.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached autora_experiment_runner_experimentation_manager_firebase-1.1.5-py3-none-any.whl.metadata (2.8 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
Using cached autora_experiment_runner_firebase_prolific-1.0.1-py3-none-any.whl (5.4 kB)
Using cached autora_experiment_runner_experimentation_manager_firebase-1.1.5-py3-none-any.whl (6.6 kB)
Using cached autora_experiment_runner_recruitment_manager_prolific-1.0.1-py3-none-any.whl (8.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 3.8 MB/s  0:00:03 eta 0:00:01
Using cached hyperframe-6.1.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 4.2 MB/s  0:00:00 eta 0:00:01
Using cached pyasn1_mo

## 2. Imports

In [2]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection
from autora.experimentalist.random import pool as random_pooler, sample as random_sampler
from autora.experiment_runner.firebase_prolific import firebase_prolific_runner
from autora.theorist.darts.regressor import DARTSRegressor

import pandas as pd

/opt/anaconda3/envs/autora_gui/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Component definitions

In [3]:
# Random Pooler
@on_state()
def random_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=random_pooler(variables, num_samples=5, replace=True))

In [4]:
# Random Sampler
@on_state()
def random_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sampler(conditions=conditions, num_samples=num_samples, replace=False))

In [5]:
# Firebase Prolific Runner
@on_state()
def firebase_prolific_runner_on_state(conditions: pd.DataFrame) -> Delta:
    runner = firebase_prolific_runner(sleep_time=30, study_completion_time=10)
    assert runner.run is not None
    return Delta(experiment_data=runner.run(conditions=conditions, exclude_studies=["default"], approve_no_code=True))

In [6]:
# DARTS Regressor
darts_regressor_on_state = estimator_on_state(DARTSRegressor(batch_size=64, num_graph_nodes=2, output_type="real", classifier_weight_decay=0.01, darts_type="original", param_updates_per_epoch=10, param_updates_for_sampled_model=100, param_learning_rate_max=0.025, param_learning_rate_min=0.01, param_momentum=0.9, arch_updates_per_epoch=1, arch_learning_rate_max=0.003, arch_weight_decay=0.0001, arch_weight_decay_df=0.0003, arch_weight_decay_base=0, arch_momentum=0.9, fair_darts_loss_weight=1, max_epochs=10, grad_clip=5, primitives=["none", "add", "subtract", "linear", "linear_logistic", "linear_relu"], train_classifier_coefficients=False, train_classifier_bias=False, sampling_strategy="max"))

## 4. Run the workflow

In [7]:
# Variables are created and governed by the experiment runner
runner = firebase_prolific_runner(sleep_time=30, study_completion_time=10)
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Main experiment loop (10 cycles)
for i in range(10):
    print(f'Cycle {i}')

    # Random Pooler
    state = random_pooler_on_state(state)

    # Random Sampler
    state = random_sampler_on_state(state, num_samples=1)

    # Firebase Prolific Runner
    state = firebase_prolific_runner_on_state(state)

    # DARTS Regressor
    state = darts_regressor_on_state(state)


print("Workflow completed!")
state

AttributeError: 'function' object has no attribute 'variables'